# Sleep + DXA study

#### Load Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
demographics_df = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/DXA_demographics.xlsx')
demographics_df.columns = demographics_df.columns.str.lower().str.strip()
demographics_df.rename(columns={'dxa id': 'dxa_id', 'profiler data': 'profiler_data', 'dxa data (y/n)': 'dxa_data', 'gender': 'sex'}, inplace=True)
demographics_df.drop(['female', 'male'], axis=1, inplace=True)

demographics_df.head()

,dxa_id,profiler_data,dxa_data,age,bmi,sex,race,ethnicity_hispaniclatinx
0,DXA001,DXA001,Y,19,20.7,Female,White or Caucasian,NaN
1,DXA002,DXA002,N,18,20.4,Male,Asian,NaN
2,DXA004,DXA004,Y,23,30.0,Male,Asian,NaN
3,DXA010,DXA010,Y,19,28.5,Female,Asian,NaN
4,DXA011,DXA011,Y,21,22.6,Female,Middle Eastern,NaN


In [31]:
body_comp_df = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/DXA_body_comp_data.xlsx')
body_comp_df.columns = body_comp_df.columns.str.lower().str.strip()
body_comp_df.drop(['id.1'], axis=1, inplace=True)
body_comp_df.rename(columns={'dxa id':'dxa_id', 'height': 'height_in', 'weight':'weight_lbs', 'lean_mass':'lean_mass_lbs', 'fat_mass':'fat_mass_lbs'
                             }, inplace=True)

bodcomp_cols_to_keep = ['dxa_id',
 'id',
 'sex',
 'age',
 'height_in',
 'weight_lbs',
 'total_mass',
 'lean_mass_lbs',
 'fat_mass_lbs',
 'bmi',
 'total_fat_lbs',
 'total_bmd',
 'total_bmc',
 'total_bmc_lbs',
 'est_visceral_adipose_tissue_vol',
 'est_visceral_adipose_tissue_mass',
 'resting_metabolic_rate',
 'relative_skeletal_muscle_index',
]
body_comp_df_filtered = body_comp_df[bodcomp_cols_to_keep].copy()

body_comp_df_filtered.head()

,dxa_id,id,sex,age,height_in,weight_lbs,total_mass,lean_mass_lbs,fat_mass_lbs,bmi,total_fat_lbs,total_bmd,total_bmc,total_bmc_lbs,est_visceral_adipose_tissue_vol,est_visceral_adipose_tissue_mass,resting_metabolic_rate,relative_skeletal_muscle_index
0,DXA001,1,F,20.8,65.0,120.5,120.90,87.70,28.20,20.1,28.20,1.128,2275.0,5.0,3.34,0.11,1241.0,7.12
1,DXA002,2,NaN,NaN,NaN,NaN,124.00,95.40,23.12,NaN,NaN,1.230,2735.6,NaN,NaN,NaN,NaN,NaN
2,DXA004,4,M,23.8,71.0,206.5,206.00,122.40,76.60,28.8,76.60,1.229,3159.0,7.0,80.44,2.74,1569.0,8.77
3,DXA008,8,F,21.8,64.0,164.0,163.60,88.50,70.00,28.2,70.00,1.208,2332.0,5.1,20.08,0.68,1250.0,7.43
4,DXA010,10,F,19.8,63.0,126.5,127.15,82.03,39.92,22.4,39.92,1.182,2356.8,5.2,7.99,0.27,1192.0,7.01


In [45]:
dietary_intake_df = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/DXA_DietaryScreener_ExpectedIntake_df.xlsx', header=1)
dietary_intake_df.columns = dietary_intake_df.columns.str.lower().str.strip()
dietary_intake_df.rename(columns={'gender': 'sex'}, inplace=True)

dietary_intake_df.tail()

,respondent_id,dexa_id,age,sex,expected_fruit_cups,expected_veg_cups,expected_fruit_vegatable_cups,expected_sugardrinks_cups,expected_diary_cups,expected_added sugar_teaspoons,expected_wholegrain_ounces,expected_fiber_grams,expected_calcium_milligrams
93,DXA094,DXA174,21,Female,0.994979,1.558585,2.553564,4.420968,1.210285,14.266775,0.461103,15.251570,850.303621
94,DXA095,DXA176,25,Male,2.010274,2.684385,4.694659,7.111202,2.372610,17.279267,1.454100,30.605196,1433.219253
95,DXA096,DXA177,24,Male,2.010274,2.302109,4.312383,4.772110,1.873566,12.882040,0.464675,26.054356,1140.368907
96,DXA097,DXA178,20,Male,1.215615,2.088487,3.304102,4.772110,1.088255,10.063610,0.608903,17.818972,893.942792
97,DXA098,DXA180,20,Female,0.994979,1.086433,2.081412,3.573933,2.146588,9.989890,1.314481,20.940021,1335.894838


In [72]:
IPAQ_df = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/DXA_IPAQ-SF_df.xlsx')
IPAQ_df.columns = IPAQ_df.columns.str.lower().str.strip()
IPAQ_df['dxa_id'] = IPAQ_df['subjectid'].apply(lambda x: f'DXA{x:03d}')

IPAQ_df.head()

,subjectid,exercise_mod_cardio,exercise_vig_cardio,exercise_strength,exercise_intensity,dxa_id
0,1,4.0,4.0,1.0,2.0,DXA001
1,2,4.0,2.0,0.0,2.0,DXA002
2,3,5.0,3.0,4.0,1.0,DXA003
3,4,1.0,0.0,0.0,2.0,DXA004
4,5,0.0,1.0,6.0,1.0,DXA005


In [69]:
DXA_sleep_nocturnal_df = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/DXA_Sleep_nocturnal_actigraphy_scored_df.xlsx')
DXA_sleep_nocturnal_df.columns = DXA_sleep_nocturnal_df.columns.str.lower().str.strip()
DXA_sleep_nocturnal_df.columns.str.replace('', '_')
DXA_sleep_nocturnal_df.rename(columns={'passcode': 'dxa_id', 'eff':'efficiency', 'night': 'night_of_week','interval#':'day_count', 
                                       }, inplace=True)
DXA_sleep_nocturnal_df.drop([0], axis=0, inplace=True)
DXA_sleep_nocturnal_df.reset_index(drop=True, inplace=True)

DXA_sleep_nocturnal_df.head()

,subjectid,dxa_id,interval type,day_count,night_of_week,start date,start day,start time,end date,end day,...,waketime,%wake,#wakebouts,avg wake b,sleep time,% sleep,#sleepbouts,avgsleepb,fragmentation,flag_actigraph
0,1,DXA001,Sleep,1,Friday,2019-09-20 00:00:00,Fri,23:21:00,2019-09-21 00:00:00,Sat,...,28,5.44,19,1.47,487,94.56,20,24.35,20.28,NaN
1,1,DXA001,Sleep,2,Saturday,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EXCLUDED - >15% of the day off-wrist
2,1,DXA001,Sleep,3,Sunday,2019-09-22 00:00:00,Sun,21:42:00,2019-09-23 00:00:00,Mon,...,38,5.86,27,1.41,610,94.14,28,21.79,20.75,NaN
3,1,DXA001,Sleep,4,Monday,2019-09-23 00:00:00,Mon,22:21:00,2019-09-24 00:00:00,Tue,...,56,9.86,26,2.15,512,90.14,27,18.96,28.54,NaN
4,1,DXA001,Sleep,5,Tuesday,2019-09-25 00:00:00,Wed,00:29:00,2019-09-25 00:00:00,Wed,...,20,4.25,13,1.54,451,95.75,14,32.21,17.8,NaN


In [89]:
sleep_profiler_9h.dtypes

condition                     object
dxa_id                        object
processing/edit_timestamp     object
night                         object
studydate                     object
                              ...   
aveimpreog                   float64
aveimpeeg                    float64
aveimpc                      float64
noepochs                     float64
nodiffhigh                   float64
Length: 120, dtype: object

In [ ]:
sleep_profiler_9h = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/Sleep_Profiler_9h_df.xlsx')
sleep_profiler_9h.columns = sleep_profiler_9h.columns.str.lower().str.strip().str.replace(' ', '_')
sleep_profiler_9h.rename(columns= {'lastname':'condition', 'firstname':'dxa_id'}, inplace=True)
sleep_profiler_9h['condition'] = sleep_profiler_9h['condition'].replace('9 hr TIB', '9h')

sleep_profiler_9h_df = sleep_profiler_9h.round(3)
sleep_profiler_9h_df.head()

,condition,dxa_id,processing/edit_timestamp,night,studydate,studytime,recordingtime,sleeptime,sleepefficiency,sleeponset,...,nohighimpleog,nohighimpreog,nohighimpeeg,nohighimpc,aveimpleog,aveimpreog,aveimpeeg,aveimpc,noepochs,nodiffhigh
0,9h,DXA001,2019-10-09 10:39:00,1,2019-10-07 23:08:18,9.133056,9.1,8.641666,94.963,3,...,1.0,0.0,1.0,37.0,11.158,2.026,2.553,245.500,1093.0,0.0
1,9h,DXA002,2019-09-30 12:40:00,1,2019-09-29 01:56:49,9.068055,8.983334,7.441667,82.839,13,...,30.0,24.0,13.0,39.0,93.538,40.385,79.308,215.692,1086.0,85.0
2,9h,DXA004,2019-10-10 11:33:00,1,2019-10-09 23:30:46,9.338056,9.3,8.258333,88.799,4,...,0.0,0.0,0.0,42.0,6.738,6.048,3.643,252.000,1117.0,0.0
3,9h,DXA008,2019-10-29 09:20:00,1,2019-10-26 01:29:36,9,8.958333,6.683333,74.605,25,...,23.0,15.0,6.0,37.0,17.216,17.000,13.297,252.000,1078.0,0.0
4,9h,DXA010,2019-10-21 11:02:00,1,2019-10-18 23:06:19,8.898889,8.858334,7.475,84.384,11,...,26.0,12.0,13.0,43.0,54.545,15.841,21.932,215.773,1065.0,0.0


In [ ]:
sleep_profiler_normal = pd.read_excel('/Users/thomasgooding/Desktop/Sleep_DXA_study/Sleep_Profiler_normal_sleep_df.xlsx')
sleep_profiler_normal.columns = sleep_profiler_normal.columns.str.lower().str.strip().str.replace(' ', '_')
sleep_profiler_normal.rename(columns= {'lastname':'condition', 'firstname':'dxa_id'}, inplace=True)

sleep_profiler_normal_df = sleep_profiler_normal.round(3)
sleep_profiler_normal.head()

,condition,dxa_id,processing/edit_timestamp,night,studydate,studytime,recordingtime,sleeptime,sleepefficiency,sleeponset,...,nohighimpleog,nohighimpreog,nohighimpeeg,nohighimpc,aveimpleog,aveimpreog,aveimpeeg,aveimpc,noepochs,nodiffhigh
0,Normal,DXA001,2019-09-27 10:20:00,1,2019-09-26 22:41:54,9.317223,9.283334,8.691667,93.62656,13,...,0.0,2.0,0.0,39.0,4.871795,5.358974,4.769231,252.0000,1116.0,0.0
1,Normal,DXA002,2019-09-26 10:39:00,1,2019-09-26 02:17:07,7.213056,7.191667,6.058333,84.24102,22,...,29.0,10.0,2.0,28.0,27.700000,16.000000,7.266667,235.7333,863.0,0.0
2,Normal,DXA004,2019-10-04 12:26:00,1,2019-10-04 01:11:58,5.814722,5.783333,5.358333,92.65129,11,...,0.0,0.0,0.0,25.0,5.120000,5.360000,3.600000,252.0000,695.0,0.0
3,Normal,DXA008,2019-10-23 11:37:00,1,2019-10-23 01:10:17,6.759167,6.741667,5.466667,81.08776,30,...,14.0,0.0,0.0,31.0,16.000000,10.741940,8.387096,252.0000,809.0,0.0
4,Normal,DXA010,2019-10-15 09:38:00,1,2019-10-15 00:27:19,6.479445,6.458333,5.8,89.80645,22,...,17.0,3.0,0.0,28.0,17.750000,12.107140,9.928572,252.0000,775.0,0.0


In [122]:
EEG_df = pd.concat([sleep_profiler_normal, sleep_profiler_9h_df], axis=0, keys = 'dxa_id'


)

EEG_df.tail()

/var/folders/7d/370678cs0jzdr2slyl3k7m2w0000gn/T/ipykernel_38198/73933723.py:1: FutureWarning: The behavior of pd.concat with len(keys) != len(objs) is deprecated. In a future version this will raise instead of truncating to the smaller of the two sequences
  EEG_df = pd.concat([sleep_profiler_normal, sleep_profiler_9h_df], axis=0, keys = 'dxa_id'


condition     dxa_id processing/edit_timestamp            night  \
x 68         9h     DXA173       2025-03-04 19:24:00                1   
  69         9h     DXA176       2025-04-21 17:42:00                1   
  70         9h     DXA177       2025-05-05 16:24:00                1   
  71  421000348  220400280               01.02.93.20    9 hr TIB redo   
  72  421000356  250100024               01.02.93.20  9 hr TIB redo 2   

                studydate            studytime recordingtime  \
x 68  2025-03-04 00:52:40             8.551111         8.525   
  69  2025-04-19 22:19:36              10.8025      10.63333   
  70  2025-05-03 23:32:59             9.140833          8.95   
  71               DXA178  2005-01-01 00:00:00             M   
  72               DXA180  2005-01-01 00:00:00             F   

                sleeptime sleepefficiency  \
x 68             7.166667          84.066   
  69             9.716666          91.379   
  70             8.441667           94.32   
  71  2025-06-17 16:14:00             1.0   
  72  2025-09-04 15:12:00             1.0   

                                            sleeponset  ... nohighimpleog  \
x 68                                                 3  ...           0.0   
  69                                                11  ...           0.0   
  70                                                 5  ...           0.0   
  71  20250617_161422____20250617_161322_0000421000348  ...           NaN   
  72  20250904_151209____20250904_151110_0000421000356  ...           NaN   

     nohighimpreog nohighimpeeg nohighimpc aveimpleog aveimpreog aveimpeeg  \
x 68           0.0          2.0       35.0      6.229      4.600    16.143   
  69           0.0          0.0       44.0      4.614      5.545     3.659   
  70           0.0          0.0       37.0      5.595      6.892     4.324   
  71           NaN          NaN        NaN        NaN        NaN       NaN   
  72           NaN          NaN        NaN        NaN        NaN       NaN   

      aveimpc noepochs nodiffhigh  
x 68  252.429   1024.0        0.0  
  69  252.000   1294.0        0.0  
  70  252.000   1094.0        0.0  
  71      NaN      NaN        NaN  
  72      NaN      NaN        NaN  

[5 rows x 120 columns]